### Explanation of location geojson

GeoJSON is composed of "features" which each represent a power line  

The geometry is linestring because its line consisting of ordered coordinate points  

Properties shows what type the object is (major or minor power line) and voltage (the others can be ignored)

```json
  {
    "type": "Feature",
    "geometry": {
      "type": "LineString",
      "coordinates": [
        [
          -92.2619846,
          39.0210987
        ],
        [
          -92.2613601,
          39.0211232
        ],
        [
          -92.2614052,
          39.0205361
        ]
      ]
    },
    "properties": {
      "cables": "3",
      "frequency": "60",
      "power": "line",
      "voltage": "69000",
      "wires": "single"
    }
  },
  ```

### Importing data from overpass

line: major overhead transmission lines  
minor_line: smaller overhead distribution lines

In [5]:
import requests, json

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
query = """
[out:json][timeout:60];

area["name"="Columbia"]["boundary"="administrative"]->.sc;

(
  way["power"="line"](area.sc);
  way["power"="minor_line"](area.sc);
);

out geom;
"""
resp = requests.post(OVERPASS_URL, data=query, headers={"Content-Type":"text/plain"})
resp.raise_for_status()
data = resp.json()         


Converting into GeoJSON

In [6]:
features = []

for el in data.get("elements", []):
    if el["type"] == "way" and "geometry" in el:
        coords = [[pt["lon"], pt["lat"]] for pt in el["geometry"]]

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            },
            "properties": el.get("tags", {})
        }

        features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open("south_carolina_power_lines.geojson", "w") as f:
    json.dump(geojson, f, indent=2)

print("Saved GeoJSON with", len(features), "features")

Saved GeoJSON with 386 features


Calculate proximity to trees

In [ ]:
# geojson of tree canopies
tree_file = ""

# 